## Setup

In [1]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
import scipy
import yaml
from sklearn.metrics import r2_score

## User Options

In [ ]:
parent_dir = r"/Volumes/T7/Image Analysis/Ester Bond Formation/" 
date_and_reactor = r"mm-dd-yy-Reactor" # Change date/reactor
file_path_string = parent_dir + date_and_reactor + r"/output/data/__/average/data.csv" # Choose yellow or white
my_df = pd.read_csv(file_path_string)

rows = 8        # Change to num rows
cols = 12       # Change to num cols
num_refs = 4    # Change to num refs

# Replace with chemical names
group3 = ['col 1', 'col 2', 'col 3', 'col 4', 'col 5', 'col 6', 'col 7', 'col 8', 'col 9', 'col 10', 'col 11', 'col 12']
group = group3          # Choose grouping grouping
group_names = "group3"  # match above


acid_chloride = ["2,4-difluorobenzoyl chloride"] # Change acid chloride
initial_conc = 4
rxnOrder = 1

lnA = [-1.609437912,
-2.995732274,
-4.382026635,
-5.768320996,
-7.154615357,
-8.540909718,
-9.927204079,
-11.31349844]

reagent_stock_concs = [0.4,
0.1,
0.025,
0.00625,
0.0015625,
0.000390625,
0.00009765625,
0.0000244140625]

df_filepath = parent_dir + r"\\Times and Concs Dataframes\\" + group_names + " + " + str(acid_chloride[0]) + " times_and_concs.xlsx"
rate_filepath = parent_dir + r"\\Rates Dataframes\\" + group_names + " + " + str(acid_chloride[0]) + " rates.xlsx"
time_array = my_df.iloc[0,7:].to_numpy()

## Finding Inflection Points

In [ ]:
inflection_points = []

def process(array):
    array_max = np.max(array)
    array_min = np.min(array)
    infl_point = array_min + (array_max - array_min) * 0.5

    # Find the index where the array first reaches or exceeds the midpoint
    greater_than_index = np.where(array >= infl_point)[0][0]
    time_at_inflection = time_array[greater_than_index]

    # Splitting the array around the inflection point for plotting
    array_1_x = time_array[:greater_than_index + 1]
    array_1_y = array[:greater_than_index + 1]
    array_2_x = time_array[greater_than_index:]
    array_2_y = array[greater_than_index:]

    return array_1_x, array_1_y, array_2_x, array_2_y, time_at_inflection, greater_than_index

fig, axs = plt.subplots(rows, cols, figsize=(2*cols, 1.7*rows))

intensity_arrays = my_df.iloc[(1+2*num_refs):, 7:].to_numpy()

for i, array in enumerate(intensity_arrays):
    # Locate the subplot row and column
    ax = axs[i // cols, i % cols]
    array_1_x, array_1_y, array_2_x, array_2_y, time_at_inflection, greater_than_index = process(array)
    inflection_points.append(time_at_inflection)

    # Plot data before and after the inflection point
    ax.plot(array_1_x, array_1_y, marker='x', lw=0.5, ms=2.0, color="blue")
    ax.plot(array_2_x, array_2_y, marker='x', lw=0.5, ms=2.0, color="yellow")
    
    # Mark the inflection point
    ax.plot(time_at_inflection, array[greater_than_index],
            marker='*', lw=0.5, ms=15.0, color="red")
    
    ax.set_title(f"Array ({i//cols+1}, {i%cols+1})")
    ax.set_xlabel('Time')
    ax.set_ylabel('Intensity')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

## Tabulating Inflection Points

In [ ]:
row_arrays = np.array_split(inflection_points,8)
new_arrays = []
for entry in row_arrays:
    new_arrays.append(np.array_split(entry,2))
array_points = np.array(row_arrays)
df_points = []
df_points = pd.DataFrame(array_points)
#df_points = df_points.reindex(index=[7, 6, 5, 4, 3, 2, 1, 0])
#df_points = df_points.drop([8, 7, 5, 3, 1], axis=1)
df_points



## Plotting Points

In [ ]:
from sklearn.metrics import r2_score

# Assuming you have a DataFrame 'df' with 12 columns (X1, X2, ..., X12)
# Replace 'X1', 'X2', ..., 'X12' with the actual column names from your DataFrame

# Specify start and end indices for each column
plot_ranges = [(0,8), (0, 8), (0,8), (0,8),(0,8), (0, 8), (0,8), (0,8), (0,8), (0, 8), (0,8), (0,8)]
#plot_ranges = [(2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7), (2,7)]

reg_coefs = []

def array_plot_with_r2(X, y, i, start_idx, end_idx, ax):
    # Slice X and y based on start and end indices
    X_slice = X[start_idx:end_idx]
    y_slice = y[start_idx:end_idx]

    # Fit a linear regression model
    reg = LinearRegression().fit(X_slice.values.reshape(-1, 1), y_slice)

    # Predict Y values using the linear regression model
    y_pred = reg.predict(X_slice.values.reshape(-1, 1))

    # Calculate R^2 value
    r2 = r2_score(y_slice, y_pred)

    # Create scatter plot
    ax.scatter(X_slice, y_slice, label=f'R2 = {r2:.4f}')
    ax.plot(X_slice, y_pred, color='red', linestyle='--', lw=0.8,label= reg.coef_[0])
    reg_coefs.append(reg.coef_[0])

    # Customize plot
    ax.set_xlabel(f'Time')
    ax.set_ylabel('1/Concentration')
    ax.set_title(str(group[i]) + "+" + str(acid_chloride[0]))
    ax.legend()

# Create a rowsxcols grid of plots
numrows = 4
numcols = 3
fig, axs = plt.subplots(numrows, numcols, figsize=(numcols*7, numrows*5))

for i, (start, end) in enumerate(plot_ranges):
    row, col = divmod(i, numcols)
    array_plot_with_r2(df_points[i], lnA, i, start, end, axs[row, col])

plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame(index = reagent_stock_concs)
for i in range(cols):
    output_list = [""]*rows
    output_times = [""]*rows
    output_list[plot_ranges[i][0]:plot_ranges[i][1]] = df_points[i][plot_ranges[i][0]:plot_ranges[i][1]].tolist()
    output_times[plot_ranges[i][0]:plot_ranges[i][1]] = lnA[plot_ranges[i][0]:plot_ranges[i][1]]
    df[(str(group[i])+ "+" + str(acid_chloride[0])+" times")] = output_list
    df[(str(group[i])+ "+" + str(acid_chloride[0])+" ln[A]")] = output_times
df

In [ ]:
df.to_excel(df_filepath)

In [ ]:
row_index = acid_chloride
rates_df = pd.DataFrame(index = row_index)
for i in range(cols):
    rates_df[str(group[i])] = reg_coefs[i]
rates_df

In [ ]:
rates_df.to_excel(rate_filepath)